In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os

# -------------------- SETTINGS --------------------
DATA_PATH = "gesture_data"  # Folder to save collected data
NUM_CLASSES = 6             # 0-5
SAMPLES_PER_CLASS = 500     # Minimum samples per gesture
# ---------------------------------------------------

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# Make folders for each class
for i in range(NUM_CLASSES):
    class_path = os.path.join(DATA_PATH, str(i))
    os.makedirs(class_path, exist_ok=True)

cap = cv2.VideoCapture(0)

# Sample counters for each class
counters = {i: len(os.listdir(os.path.join(DATA_PATH, str(i)))) for i in range(NUM_CLASSES)}

print("Press keys 0-5 to record gestures for that number.")
print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1)  # Mirror image
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    # Draw landmarks
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    cv2.putText(frame, "Samples: " + str(counters), (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("Gesture Data Collection", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    # Record sample if 0-5 key is pressed
    if key in [ord(str(i)) for i in range(NUM_CLASSES)]:
        class_id = int(chr(key))
        if counters[class_id] < SAMPLES_PER_CLASS:
            if result.multi_hand_landmarks:
                hand_landmarks = result.multi_hand_landmarks[0]
                # Flatten 21 landmarks x 3 coordinates
                landmarks = []
                for lm in hand_landmarks.landmark:
                    landmarks.extend([lm.x, lm.y, lm.z])
                landmarks = np.array(landmarks)
                # Save as CSV
                filename = os.path.join(DATA_PATH, str(class_id), f"{counters[class_id]}.csv")
                np.savetxt(filename, landmarks, delimiter=",")
                counters[class_id] += 1
                print(f"Saved gesture {class_id}: {counters[class_id]}/{SAMPLES_PER_CLASS}")
            else:
                print("No hand detected! Try again.")

cap.release()
cv2.destroyAllWindows()


2025-12-05 11:50:10.435505: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1764910233.938906  290197 gl_context.cc:357] GL version: 2.1 (2.1 INTEL-18.8.16), renderer: Intel Iris Pro OpenGL Engine
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1764910233.961425  290610 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764910233.978129  290610 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Press keys 0-5 to record gestures for that number.
Press 'q' to quit.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
